- Parcial práctico
- Alejandra Diaz
- Estadística
- Machine Learning con PySpark y Docker


Instalar dependencias y librerias

In [1]:
# instalacion
!pip install pyspark imbalanced-learn

# Librerias principales
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct,sum


from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

from imblearn.over_sampling import SVMSMOTE

import pandas as pd
import os
import shutil

from google.colab import files





In [2]:
spark = SparkSession.builder \
    .appName("FraudeDetection") \
    .getOrCreate()

Carga de información

In [3]:
uploaded = files.upload()

Saving train.csv to train.csv
Saving test.csv to test.csv


---
# EDA

---

Estructura y visualización inicial

In [4]:
train = spark.read.csv("train.csv", header=True, inferSchema=True)
test  = spark.read.csv("test.csv",  header=True, inferSchema=True)

train.printSchema()
test.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [5]:
train.show(5)
test.show(5)

+--------+-------------------+------------------+------------------+------------------+-----------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+-------------------+-------+-----+
|    Time|                 V1|                V2|                V3|                V4|               V5|                V6|                V7|                 V8|                V9|               V10|               V11|               V12|               V13|               V14|               V15|               V16|               V17|               V18|               V19|                V20|               V21|               V22|         

Dimensiones

In [7]:
print("Train filas:", train.count(), "columnas:", len(train.columns))
print("Test filas:", test.count(), "columnas:", len(test.columns)-1)

Train filas: 199364 columnas: 31
Test filas: 42721 columnas: 30


Valores faltantes

In [8]:
train.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in train.columns
]).show()

test.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in test.columns
]).show()

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|   0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|     0|    0|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+

+---+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+
| id|Time| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|
+---+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-

Validación de transacciones duplicadas

In [12]:
test.groupBy("Id").count().filter("count > 1").show()

+---+-----+
| Id|count|
+---+-----+
+---+-----+



Mapeo global

In [13]:
train.describe().show()
test.describe().show()

+-------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|summary|             Time|                  V1|                  V2|                  V3|                  V4|                  V5|                  V6|                  V7|                  V8|                  V9|               V10|                 V11|                 V12|                 V13|                 V14|                 V15|       

Comparación de campos

In [14]:
print("Columnas iguales:", train.columns == test.columns)

Columnas iguales: False


---
Variable objetivo
---

Distribución de clases

In [18]:
total = train.count()

train.groupBy("Class").count() \
    .withColumn("proporcion", col("count") / total / (1/100)) \
    .show()

+-----+------+-------------------+
|Class| count|         proporcion|
+-----+------+-------------------+
|    1|   344|0.17254870488152324|
|    0|199020|  99.82745129511848|
+-----+------+-------------------+



In [5]:
target = "Class"
features = [c for c in train.columns if c != target]

Por el desbalanceo de datos se hace necesario generar datos sintéticos de la clase minoritaria y así tratar de evitar que el modelo memorice los datos y trate de aprender las relaciones existentes en los datos.

In [6]:
train_pd = train.select(features + [target]).toPandas()

X = train_pd[features]
y = train_pd[target]

In [7]:
svm_smote = SVMSMOTE(
    sampling_strategy=0.15,
    random_state=5593
)

X_res, y_res = svm_smote.fit_resample(X, y)

In [8]:
y_res = pd.Series(y_res, name="Class")
train_balanced_pd = pd.concat([X_res, y_res], axis=1)

train_balanced = spark.createDataFrame(train_balanced_pd)

In [9]:
train_balanced.groupBy("Class").count().show()

+-----+------+
|Class| count|
+-----+------+
|    0|199020|
|    1| 29853|
+-----+------+



In [10]:
train_split, val = train_balanced.randomSplit([0.75, 0.25], seed=5593)

In [11]:
assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)

if "features" not in train_split.columns:
    train_split = assembler.transform(train_split)

if "features" not in val.columns:
    val = assembler.transform(val)

Modelo de regresión logística

In [12]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="Class"
)

model_lr = lr.fit(train_split)
pred_lr = model_lr.transform(val)

Modelo de regresión Lasso

In [13]:
lr_lasso = LogisticRegression(
    featuresCol="features",
    labelCol="Class",
    regParam=0.01,
    elasticNetParam=1
)

model_lasso = lr_lasso.fit(train_split)
pred_lasso = model_lasso.transform(val)

Modelo Random Forest

In [14]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="Class",
    numTrees=300,
    maxDepth=10
)

model_rf = rf.fit(train_split)
pred_rf = model_rf.transform(val)

In [15]:
auc_eval = BinaryClassificationEvaluator(labelCol="Class", metricName="areaUnderROC")
pr_eval  = BinaryClassificationEvaluator(labelCol="Class", metricName="areaUnderPR")

función de evaluación y consolidación de resultados

In [16]:
def evaluar(nombre, pred):
    auc = auc_eval.evaluate(pred)
    pr  = pr_eval.evaluate(pred)

    rdd = pred.select("prediction", "Class").rdd
    metrics = MulticlassMetrics(rdd)

    precision = metrics.precision(1.0)
    recall    = metrics.recall(1.0)
    f1        = metrics.fMeasure(1.0)

    print(f"\n📊 MATRIZ DE CONFUSIÓN - {nombre}")
    print(metrics.confusionMatrix().toArray())

    return (nombre, auc, pr, f1, precision, recall)

In [19]:
pred_lr = pred_lr.withColumn("Class", col("Class").cast("double"))
pred_lasso = pred_lasso.withColumn("Class", col("Class").cast("double"))
pred_rf = pred_rf.withColumn("Class", col("Class").cast("double"))

Resultados

In [20]:
results = []

results.append(evaluar("Logistic Regression", pred_lr))
results.append(evaluar("Lasso", pred_lasso))
results.append(evaluar("Random Forest", pred_rf))


📊 MATRIZ DE CONFUSIÓN - Logistic Regression
[[4.9679e+04 2.5000e+01]
 [4.2000e+01 7.4620e+03]]

📊 MATRIZ DE CONFUSIÓN - Lasso
[[4.9686e+04 1.8000e+01]
 [3.0900e+02 7.1950e+03]]

📊 MATRIZ DE CONFUSIÓN - Random Forest
[[4.9696e+04 8.0000e+00]
 [2.7000e+01 7.4770e+03]]


Tabla resumen de los modelos entrenados

In [21]:
results_df = spark.createDataFrame(
    results,
    ["Model", "AUC", "PR-AUC", "F1 Score", "Precision", "Recall"]
)

results_df.show(truncate=False)

+-------------------+------------------+------------------+------------------+------------------+------------------+
|Model              |AUC               |PR-AUC            |F1 Score          |Precision         |Recall            |
+-------------------+------------------+------------------+------------------+------------------+------------------+
|Logistic Regression|0.999481483956452 |0.9988458243860808|0.9955306517243679|0.996660878856685 |0.9944029850746269|
|Lasso              |0.9996276584780626|0.998546583466695 |0.9777807977169259|0.9975045057535006|0.9588219616204691|
|Random Forest      |0.999404220855268 |0.9990751440094692|0.9976649542998199|0.998931195724783 |0.9964019189765458|
+-------------------+------------------+------------------+------------------+------------------+------------------+



In [30]:
if "features" not in test.columns:
    test = assembler.transform(test)

pred_test = model_rf.transform(test)

pred_test.select("prediction", "probability",'prediction').show(10)

+----------+--------------------+----------+
|prediction|         probability|prediction|
+----------+--------------------+----------+
|       0.0|[0.99980999848683...|       0.0|
|       0.0|[0.99968381703062...|       0.0|
|       0.0|[0.99973834966111...|       0.0|
|       0.0|[0.99956553025203...|       0.0|
|       0.0|[0.99982703621174...|       0.0|
|       0.0|[0.99979382582466...|       0.0|
|       0.0|[0.99982701413107...|       0.0|
|       0.0|[0.99978239790205...|       0.0|
|       0.0|[0.99982756075804...|       0.0|
|       0.0|[0.99982242737381...|       0.0|
+----------+--------------------+----------+
only showing top 10 rows


Validación en datos sin etiqueta

In [31]:
salida = pred_test.select("Id",'prediction').show(10)

+---+----------+
| Id|prediction|
+---+----------+
|  1|       0.0|
|  2|       0.0|
|  3|       0.0|
|  4|       0.0|
|  5|       0.0|
|  6|       0.0|
|  7|       0.0|
|  8|       0.0|
|  9|       0.0|
| 10|       0.0|
+---+----------+
only showing top 10 rows


In [33]:
salida = pred_test.select("Id", "prediction")
salida = salida.withColumnRenamed("prediction", "Class")
salida = salida.withColumn("Class", col("Class").cast("int"))

salida.coalesce(1).write.csv(
    "submission",
    header=True,
    mode="overwrite"
)

for file in os.listdir("submission"):
    if file.endswith(".csv"):
        shutil.copy(f"submission/{file}", "submission_final.csv")

files.download("submission_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
salida.select(countDistinct("Id")).show()

+------------------+
|count(DISTINCT Id)|
+------------------+
|             42721|
+------------------+



In [35]:
salida.groupBy("Class").agg(count("*").alias("cantidad")).show()

+-----+--------+
|Class|cantidad|
+-----+--------+
|    1|      59|
|    0|   42662|
+-----+--------+

